# 04 · Attention from scratch

Attention is the operation at the heart of the transformer. In this notebook you build it in NumPy, step by step, and test every step with check cells:

1. scaled dot-product attention,
2. a causal mask (no looking at the future),
3. multi-head attention.

Nothing is downloaded except NumPy. Read the shape comments carefully: most bugs in this code are shape bugs.

In [ ]:
import numpy as np
import lab

rng = np.random.default_rng(0)
np.set_printoptions(precision=3, suppress=True)

## 1. Scaled dot-product attention

We have $n_q$ **queries**, and $n_k$ **keys**, each key with a **value**:

| | shape | meaning |
|---|---|---|
| $Q$ | $(n_q, d_k)$ | one row per query: "what am I looking for?" |
| $K$ | $(n_k, d_k)$ | one row per key: "what do I offer?" |
| $V$ | $(n_k, d_v)$ | one row per value: "what do I pass on?" |

Every query is compared with every key by a dot product. The scores go through a softmax, row by row, and the result weights the values:

$$\mathrm{Attention}(Q, K, V) = \mathrm{softmax}\!\left(\frac{Q K^\top}{\sqrt{d_k}}\right) V$$

Shapes: $Q K^\top$ is $(n_q, n_k)$, one score per (query, key) pair. After the softmax, every row is a probability distribution over the keys. The output is $(n_q, d_v)$: for every query, a weighted average of the values.

**Why divide by $\sqrt{d_k}$?** If the components of a query and a key are independent with mean 0 and variance 1, their dot product has variance $d_k$. Without the scaling, the scores grow with the dimension, the softmax saturates (one weight near 1, the rest near 0) and its gradient becomes tiny. Dividing by $\sqrt{d_k}$ brings the variance back to 1.

In [ ]:
def softmax(x, axis=-1):
    """Softmax along one axis. Subtracting the maximum first avoids overflow and changes nothing in the result."""
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)


def scaled_dot_product_attention(Q, K, V, mask=None):
    """Q: (..., n_q, d_k), K: (..., n_k, d_k), V: (..., n_k, d_v).

    mask: boolean, broadcastable to (..., n_q, n_k). True = this query may look at this key.
    Returns (output of shape (..., n_q, d_v), weights of shape (..., n_q, n_k)).
    """
    d_k = Q.shape[-1]
    scores = Q @ np.swapaxes(K, -1, -2) / np.sqrt(d_k)      # (..., n_q, n_k)
    if mask is not None:
        scores = np.where(mask, scores, -np.inf)             # exp(-inf) = 0: forbidden keys get weight 0
    weights = softmax(scores, axis=-1)                       # each row sums to 1
    return weights @ V, weights

In [ ]:
n_q, n_k, d_k, d_v = 3, 4, 8, 5
Q = rng.standard_normal((n_q, d_k))
K = rng.standard_normal((n_k, d_k))
V = rng.standard_normal((n_k, d_v))

out, weights = scaled_dot_product_attention(Q, K, V)
print("output shape: ", out.shape)       # (3, 5)
print("weights shape:", weights.shape)   # (3, 4)
print(weights)

In [ ]:
assert out.shape == (n_q, d_v) and weights.shape == (n_q, n_k)
assert np.allclose(weights.sum(axis=-1), 1.0), "every row of the weights is a probability distribution"
assert np.all(weights >= 0)

# If all scores are equal (here: all queries are zero), every key gets the same weight,
# and the output is the plain average of the values.
uniform_out, uniform_weights = scaled_dot_product_attention(np.zeros((n_q, d_k)), K, V)
assert np.allclose(uniform_weights, 1 / n_k)
assert np.allclose(uniform_out, V.mean(axis=0))

# A query that points strongly in the direction of one key attends (almost) only to that key.
sharp_out, sharp_weights = scaled_dot_product_attention(50 * K[[2]], K, V)
assert sharp_weights[0].argmax() == 2 and sharp_weights[0, 2] > 0.99
assert np.allclose(sharp_out[0], V[2], atol=0.05)

# The scaling: with unit-variance inputs, the scores have variance close to 1 for any d_k.
for d in (4, 64, 1024):
    q, k = rng.standard_normal((2000, d)), rng.standard_normal((2000, d))
    scaled = (q * k).sum(axis=1) / np.sqrt(d)
    assert 0.8 < scaled.var() < 1.2, (d, scaled.var())
print("Scaled dot-product attention works.")

## 2. Self-attention on a sentence, and a picture of the weights

In **self-attention**, queries, keys and values all come from the same sequence: every token looks at every token. Below, each of 6 tokens gets a random vector, and the weights are shown as an image: row $i$ = how much token $i$ attends to each token (white = 1, black = 0).

In [ ]:
tokens = ["the", "cat", "sat", "on", "the", "mat"]
n, d_model = len(tokens), 16
X = rng.standard_normal((n, d_model))            # (6, 16): one vector per token

_, self_weights = scaled_dot_product_attention(X, X, X)


def show_weights(weights, scale=32):
    """Show a matrix of attention weights as a grey image, every entry enlarged to scale x scale pixels."""
    lab.show(np.kron(weights, np.ones((scale, scale))))


show_weights(self_weights)
lab.table([{"token": t, **{f"→ {u} ({j})": round(float(w), 2) for j, (u, w) in enumerate(zip(tokens, row))}} for t, row in zip(tokens, self_weights)])

The diagonal is bright: with random vectors, a token is most similar to itself ($x \cdot x = \lVert x \rVert^2 \approx d$, while $x \cdot y \approx 0$ on average). The two occurrences of "the" got *different* random vectors here, so they do not attend to each other; with real embeddings they would start from the same vector.

## 3. The causal mask

A language model predicts token $i + 1$ from tokens $1 \dots i$. While training on a whole sentence at once, position $i$ must therefore not see the positions after it. The **causal mask** is a lower-triangular matrix: `mask[i, j]` is True exactly when $j \le i$.

Masked scores are set to $-\infty$ *before* the softmax, so the remaining weights still sum to 1.

In [ ]:
def causal_mask(n):
    """(n, n) boolean matrix: True where the query position i may look at the key position j, that is j <= i."""
    return np.tril(np.ones((n, n), dtype=bool))


print(causal_mask(4).astype(int))

causal_out, causal_weights = scaled_dot_product_attention(X, X, X, mask=causal_mask(n))
show_weights(causal_weights)

In [ ]:
assert causal_mask(4).tolist() == [[True, False, False, False], [True, True, False, False], [True, True, True, False], [True, True, True, True]]
assert np.allclose(np.triu(causal_weights, k=1), 0.0), "no weight on future positions"
assert np.allclose(causal_weights.sum(axis=-1), 1.0), "rows still sum to 1"
assert np.allclose(causal_weights[0], [1, 0, 0, 0, 0, 0]), "the first token can only look at itself"

# The point of the mask: changing a later token does not change the output at earlier positions.
X_changed = X.copy()
X_changed[4] = rng.standard_normal(d_model)                       # replace token 4
changed_out, _ = scaled_dot_product_attention(X_changed, X_changed, X_changed, mask=causal_mask(n))
assert np.allclose(changed_out[:4], causal_out[:4]), "positions 0..3 must not notice the change"
assert not np.allclose(changed_out[4:], causal_out[4:]), "positions 4 and 5 do notice it"

# Without the mask, every position notices.
free_out, _ = scaled_dot_product_attention(X, X, X)
free_changed, _ = scaled_dot_product_attention(X_changed, X_changed, X_changed)
assert not np.allclose(free_changed[:4], free_out[:4])
print("The causal mask works.")

## 4. Multi-head attention

One attention operation computes one set of weights. A transformer layer runs several of them in parallel, the **heads**, each in its own smaller subspace, so that one head can follow, say, the syntax while another follows what a pronoun refers to.

With model dimension $d_\text{model}$ and $h$ heads, every head works in dimension $d_k = d_\text{model} / h$:

1. **Project.** Three learned matrices $W_Q, W_K, W_V$ of shape $(d_\text{model}, d_\text{model})$ map the input $X$ of shape $(n, d_\text{model})$ to queries, keys and values, each $(n, d_\text{model})$.
2. **Split into heads.** Reshape $(n, d_\text{model}) \to (n, h, d_k)$ and move the head axis to the front: $(h, n, d_k)$.
3. **Attend.** Scaled dot-product attention runs for all heads at once (the head axis is a batch axis): weights $(h, n, n)$, outputs $(h, n, d_k)$.
4. **Merge.** Move the head axis back and reshape: $(h, n, d_k) \to (n, h, d_k) \to (n, d_\text{model})$. This concatenates the heads' outputs.
5. **Project out.** A last learned matrix $W_O$ of shape $(d_\text{model}, d_\text{model})$ mixes the heads.

In formulas, for head $i$ with its slices of the projections:

$$\mathrm{head}_i = \mathrm{Attention}(X W_Q^{(i)},\; X W_K^{(i)},\; X W_V^{(i)}), \qquad \mathrm{MultiHead}(X) = \mathrm{Concat}(\mathrm{head}_1, \dots, \mathrm{head}_h)\, W_O$$

Here $X$ holds one token per **row**, so projections are written $X W$. The layer has $4\, d_\text{model}^2$ weights (real implementations add bias vectors; they are left out here). The number of parameters does not depend on $h$.

In [ ]:
def split_heads(x, n_heads):
    """(..., n, d_model) -> (..., n_heads, n, d_model / n_heads)"""
    *batch, n, d_model = x.shape
    assert d_model % n_heads == 0, "d_model must be divisible by the number of heads"
    x = x.reshape(*batch, n, n_heads, d_model // n_heads)     # (..., n, h, d_k)
    return np.swapaxes(x, -2, -3)                             # (..., h, n, d_k)


def merge_heads(x):
    """(..., n_heads, n, d_k) -> (..., n, n_heads * d_k). The inverse of split_heads."""
    x = np.swapaxes(x, -2, -3)                                # (..., n, h, d_k)
    *batch, n, n_heads, d_k = x.shape
    return x.reshape(*batch, n, n_heads * d_k)


def init_params(d_model, rng):
    """Four (d_model, d_model) matrices with small random entries."""
    scale = 1 / np.sqrt(d_model)
    return {name: rng.standard_normal((d_model, d_model)) * scale for name in ("W_Q", "W_K", "W_V", "W_O")}


def multi_head_attention(X, params, n_heads, mask=None):
    """X: (..., n, d_model). Returns (output (..., n, d_model), weights (..., n_heads, n, n))."""
    Q = split_heads(X @ params["W_Q"], n_heads)               # (..., h, n, d_k)
    K = split_heads(X @ params["W_K"], n_heads)               # (..., h, n, d_k)
    V = split_heads(X @ params["W_V"], n_heads)               # (..., h, n, d_k)
    heads, weights = scaled_dot_product_attention(Q, K, V, mask=mask)   # (..., h, n, d_k), (..., h, n, n)
    return merge_heads(heads) @ params["W_O"], weights        # (..., n, d_model)

In [ ]:
n_heads = 4
params = init_params(d_model, rng)

mha_out, mha_weights = multi_head_attention(X, params, n_heads, mask=causal_mask(n))
print("input:  ", X.shape)             # (6, 16)
print("output: ", mha_out.shape)       # (6, 16)
print("weights:", mha_weights.shape)   # (4, 6, 6): one (6, 6) matrix per head

# The four heads side by side (each 6 x 6), separated by a grey column.
gap = np.full((n, 1), 0.5)
show_weights(np.concatenate([part for head in mha_weights for part in (head, gap)][:-1], axis=1), scale=24)

In [ ]:
# Shapes.
assert split_heads(X, n_heads).shape == (n_heads, n, d_model // n_heads)
assert np.array_equal(merge_heads(split_heads(X, n_heads)), X), "merge_heads undoes split_heads"
assert mha_out.shape == (n, d_model) and mha_weights.shape == (n_heads, n, n)
assert np.allclose(mha_weights.sum(axis=-1), 1.0)
assert np.allclose(np.triu(mha_weights, k=1), 0.0), "the mask applies to every head"

# With one head, multi-head attention is the formula from section 1 with projected inputs.
one_out, _ = multi_head_attention(X, params, n_heads=1)
direct, _ = scaled_dot_product_attention(X @ params["W_Q"], X @ params["W_K"], X @ params["W_V"])
assert np.allclose(one_out, direct @ params["W_O"])

# Head i only uses columns i*d_k .. (i+1)*d_k of the projections.
d_head = d_model // n_heads
free_out, free_weights = multi_head_attention(X, params, n_heads)
for i in range(n_heads):
    cols = slice(i * d_head, (i + 1) * d_head)
    _, w = scaled_dot_product_attention(X @ params["W_Q"][:, cols], X @ params["W_K"][:, cols], X @ params["W_V"][:, cols])
    assert np.allclose(free_weights[i], w)

# A batch of sequences: the same function, one more axis in front.
batch = rng.standard_normal((3, n, d_model))
batch_out, batch_weights = multi_head_attention(batch, params, n_heads, mask=causal_mask(n))
assert batch_out.shape == (3, n, d_model) and batch_weights.shape == (3, n_heads, n, n)
for b in range(3):
    single, _ = multi_head_attention(batch[b], params, n_heads, mask=causal_mask(n))
    assert np.allclose(batch_out[b], single)

assert sum(p.size for p in params.values()) == 4 * d_model ** 2
print("Multi-head attention works.")

## 5. Attention does not know the order of the tokens

One more experiment. Without a mask, shuffle the rows of the input: the output is the same rows, shuffled in the same way. Attention treats its input as a *set*: nothing in the formulas depends on the position of a token.

This is why transformers add **positional information** to the token vectors before the first layer.

In [ ]:
permutation = rng.permutation(n)
shuffled_out, _ = multi_head_attention(X[permutation], params, n_heads)
assert np.allclose(shuffled_out, free_out[permutation]), "shuffling the input shuffles the output in the same way"

# With the causal mask the order does matter: position i sees a different set of tokens after shuffling.
shuffled_causal, _ = multi_head_attention(X[permutation], params, n_heads, mask=causal_mask(n))
assert not np.allclose(shuffled_causal, mha_out[permutation])
print("Without a mask, attention is permutation-equivariant.")

## Questions

1. The weights matrix has $n^2$ entries. What does that mean for the memory and time needed for a text of 1000 tokens compared with 100 tokens?
2. In `scaled_dot_product_attention`, what happens if a row of the mask is all False? Try it, and explain the result.
3. Change `n_heads` to 1, 2, 3 and 8. Which values work, and why does the number of parameters stay the same?
4. A full transformer layer adds a residual connection, layer normalisation and a small two-layer network after the attention. Sketch the shapes of a layer for $n = 6$, $d_\text{model} = 16$.